# 04 — Hyperparameter Tuning with Optuna
## EGX Directional Price Prediction — Aggressive Regularisation

### Problem Statement
Notebook 03 established a working alpha signal (val ROC-AUC ~0.54) but revealed severe memorisation in our tree-based models:

| Architecture | Overfitting Gap | Status |
|---|---|---|
| Logistic Regression | +0.04 to +0.16 | ✓ Healthy |
| Random Forest | +0.31 to +0.38 | ⚠ Moderate |
| XGBoost / LightGBM | +0.44 to +0.52 | ✗ Severe memorisation |

Root cause: our training set is only ~1,150 rows. Default hyperparameters (max_depth=5, 400 estimators, min_child_weight=1) produce trees with 32 leaf nodes that can fit approximately 35 rows per leaf — trivially memorising historical price patterns.

### Goal
Use **Optuna** with aggressive regularisation search spaces to:
1. Close the overfitting gap to **< 0.10**
2. Elevate validation ROC-AUC into the **0.56–0.60** target range
3. Diagnose whether the 2024 validation window represents a model failure or a structural **macro-regime shift** (March 2024 EGP devaluation)

### Data Split Reminder
```
Train  : 2019-01-01 → 2023-12-31  (~1,150 rows)
Val    : 2024-01-01 → 2024-12-31  (~237 rows, contains March 2024 EGP devaluation)
Test   : 2025-01-01 → present     (~348 rows — SEALED, not opened until notebook 05)
```

⚠️ **The test set is never loaded, never split, never evaluated in this notebook.**

## 1. Imports

In [1]:
from __future__ import annotations

import json
import logging
import warnings
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import mlflow
import mlflow.lightgbm
import mlflow.sklearn
import mlflow.xgboost

from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, precision_score, recall_score
from xgboost import XGBClassifier

# Silence Optuna's per-trial INFO logs — we handle progress ourselves
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
logging.getLogger("lightgbm").setLevel(logging.ERROR)

print("✓ All imports loaded.")

d:\my_projects\Egx-analyst\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ All imports loaded.


## 2. Configuration

In [2]:
# ── Feature contract ──────────────────────────────────────────────────────────
FEATURE_COLUMNS: List[str] = [
    "RSI_14",
    "MACD_Norm", "MACD_Signal_Norm", "MACD_Hist_Norm",
    "Bollinger_Width", "Bollinger_Position",
    "ATR_Pct",
    "Return_Lag_1", "Return_Lag_5", "Return_Lag_10", "Return_Lag_21",
    "Close_MA5_Ratio", "Close_MA21_Ratio",
    "Close_CV5", "Close_CV21",
    "Volume_Ratio", "Volume_Spike",
    "Day_Of_Week", "Month", "is_Ramadan",
]
TARGET_COLUMN: str = "Target"

# ── Chronological split anchors ───────────────────────────────────────────────
TRAIN_END: str = "2023-12-31"
VAL_END:   str = "2024-12-31"
REGIME_SPLIT: str = "2024-06-30"   # H1 / H2 2024 diagnostic boundary

# ── Optuna ────────────────────────────────────────────────────────────────────
N_TRIALS: int = 75          # trials per ticker — increase to 150 for production runs
RANDOM_STATE: int = 42

# ── Paths ─────────────────────────────────────────────────────────────────────
def resolve_path(candidates: List[Path]) -> Path:
    for p in candidates:
        if p.exists():
            return p
    return candidates[0]

PROCESSED_DIR = resolve_path([Path("../data/processed"), Path("data/processed")])
MODELS_DIR    = resolve_path([Path("../models"), Path("models")])
MLFLOW_DB     = resolve_path([Path("../mlflow.db"), Path("mlflow.db")])
MODELS_DIR.mkdir(parents=True, exist_ok=True)

WINNER_CONFIG_PATH = MODELS_DIR / "winner_config.json"
OPTIMIZED_HP_PATH  = MODELS_DIR / "optimized_hyperparameters.json"
ARTIFACTS_DIR      = Path("tuning_artifacts")
ARTIFACTS_DIR.mkdir(exist_ok=True)

# ── MLflow ────────────────────────────────────────────────────────────────────
MLFLOW_EXPERIMENT_NAME: str = "EGX_Hyperparameter_Tuning"

print(f"Processed data : {PROCESSED_DIR.resolve()}")
print(f"Models dir     : {MODELS_DIR.resolve()}")
print(f"MLflow DB      : {MLFLOW_DB.resolve()}")
print(f"Winner config  : {WINNER_CONFIG_PATH}")
print(f"Optuna trials  : {N_TRIALS} per ticker")
print(f"Random state   : {RANDOM_STATE}")

Processed data : D:\my_projects\Egx-analyst\data\processed
Models dir     : D:\my_projects\Egx-analyst\notebooks\models
MLflow DB      : D:\my_projects\Egx-analyst\mlflow.db
Winner config  : models\winner_config.json
Optuna trials  : 75 per ticker
Random state   : 42


## 3. Load Winner Configuration

Reads `models/winner_config.json` produced by notebook 03. This tells us which architecture won per ticker so we build the correct Optuna search space — no hardcoding.

In [3]:
def load_winner_config(path: Path) -> Dict[str, Dict[str, Any]]:
    if not path.exists():
        raise FileNotFoundError(
            f"Winner config not found at {path}.\n"
            "Run notebook 03 first to generate models/winner_config.json."
        )
    with open(path, "r") as f:
        config = json.load(f)
    return config


WINNER_CONFIG = load_winner_config(WINNER_CONFIG_PATH)

print("Winner configuration loaded:")
print("-" * 55)
for ticker, info in WINNER_CONFIG.items():
    print(
        f"  {ticker:<12}  →  {info['model']:<22}  "
        f"baseline val_roc_auc={info['val_roc_auc']:.4f}  "
        f"baseline val_f1={info['val_f1']:.4f}"
    )

TICKERS: List[str] = list(WINNER_CONFIG.keys())
print(f"\nTickers to tune: {TICKERS}")

Winner configuration loaded:
-------------------------------------------------------
  COMI_CA       →  XGBoost                 baseline val_roc_auc=0.5450  baseline val_f1=0.5517
  HRHO_CA       →  XGBoost                 baseline val_roc_auc=0.5496  baseline val_f1=0.5089
  ORWE_CA       →  LightGBM                baseline val_roc_auc=0.5489  baseline val_f1=0.5446
  SWDY_CA       →  RandomForest            baseline val_roc_auc=0.5133  baseline val_f1=0.5704
  TMGH_CA       →  LightGBM                baseline val_roc_auc=0.5209  baseline val_f1=0.4076

Tickers to tune: ['COMI_CA', 'HRHO_CA', 'ORWE_CA', 'SWDY_CA', 'TMGH_CA']


## 4. Data Loading and Chronological Split

In [4]:
def load_ticker_splits(
    ticker: str,
) -> Tuple[
    pd.DataFrame, pd.Series,   # X_train, y_train
    pd.DataFrame, pd.Series,   # X_val,   y_val   (full 2024)
    pd.DataFrame, pd.Series,   # X_val_h1, y_val_h1  (pre-devaluation)
    pd.DataFrame, pd.Series,   # X_val_h2, y_val_h2  (post-devaluation)
    pd.DatetimeIndex,           # val index (for diagnostics)
]:
    """
    Load parquet for one ticker, split chronologically.
    The test set (> VAL_END) is never extracted here.
    The val set is additionally split at REGIME_SPLIT for macro diagnostics.
    """
    path = PROCESSED_DIR / f"{ticker}_features.parquet"
    if not path.exists():
        raise FileNotFoundError(f"Parquet not found: {path}")

    df = pd.read_parquet(path).sort_index()

    # Strict date-anchored splits — no shuffling, no leakage
    train = df.loc[df.index <= TRAIN_END]
    val   = df.loc[(df.index > TRAIN_END) & (df.index <= VAL_END)]
    # Test set: df.index > VAL_END — intentionally not extracted

    def xy(subset: pd.DataFrame) -> Tuple[pd.DataFrame, pd.Series]:
        return (
            subset[FEATURE_COLUMNS].copy(),
            subset[TARGET_COLUMN].astype(int).copy(),
        )

    X_train, y_train = xy(train)
    X_val,   y_val   = xy(val)

    # Macro-regime sub-splits of the validation set
    val_h1 = val.loc[val.index < REGIME_SPLIT]
    val_h2 = val.loc[val.index >= REGIME_SPLIT]
    X_val_h1, y_val_h1 = xy(val_h1)
    X_val_h2, y_val_h2 = xy(val_h2)

    return X_train, y_train, X_val, y_val, X_val_h1, y_val_h1, X_val_h2, y_val_h2, val.index


# Preview splits
print(f"{'Ticker':<12} {'Train':>8} {'Val':>6} {'Val H1':>8} {'Val H2':>8}")
print("-" * 46)
for ticker in TICKERS:
    X_tr, y_tr, X_v, y_v, X_h1, y_h1, X_h2, y_h2, _ = load_ticker_splits(ticker)
    print(f"{ticker:<12} {len(X_tr):>8,} {len(X_v):>6,} {len(X_h1):>8,} {len(X_h2):>8,}")

Ticker          Train    Val   Val H1   Val H2
----------------------------------------------
COMI_CA         1,151    237      110      127
HRHO_CA         1,151    237      110      127
ORWE_CA         1,154    237      110      127
SWDY_CA         1,154    237      110      127
TMGH_CA         1,155    237      110      127


## 5. MLflow Setup

In [5]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")

experiment = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT_NAME)
if experiment is None:
    experiment_id = mlflow.create_experiment(MLFLOW_EXPERIMENT_NAME)
    print(f"✓ Created MLflow experiment: '{MLFLOW_EXPERIMENT_NAME}' (id={experiment_id})")
else:
    experiment_id = experiment.experiment_id
    print(f"✓ Resuming MLflow experiment: '{MLFLOW_EXPERIMENT_NAME}' (id={experiment_id})")

print(f"  Tracking URI : {mlflow.get_tracking_uri()}")
print("  → 'mlflow ui --port 5001' to separate from notebook 03 experiment.")

✓ Resuming MLflow experiment: 'EGX_Hyperparameter_Tuning' (id=2)
  Tracking URI : sqlite:///mlflow.db
  → 'mlflow ui --port 5001' to separate from notebook 03 experiment.


## 6. Optuna Objective Functions

### Design principle: Why these search space boundaries?

With only **1,150 training rows** and **20 features**, a tree with `max_depth=5` produces up to **32 leaf nodes**, meaning each leaf covers ~36 rows on average — trivial to memorise. The boundaries below force trees to stay simple enough that they must learn generalisable patterns:

- `max_depth 2–5`: Leaf count stays between 4 and 32. At depth=2 the model can only capture second-order interactions — much harder to overfit.
- `min_child_weight 10–50` (XGBoost) / `min_child_samples 30–150` (LightGBM): Forces each leaf to represent at least 30–150 training rows. This is the single most effective regulariser for small datasets.
- `reg_alpha 0.1–10` / `reg_lambda 1–20`: L1/L2 penalties that shrink leaf weights toward zero when evidence is weak.
- `gamma 0–5` (XGBoost): Minimum loss reduction required to make a split — effectively a split gate.
- `num_leaves 8–31` (LightGBM): Hard cap on total leaves regardless of depth.

In [6]:
def build_xgboost_model(trial: optuna.Trial) -> XGBClassifier:
    """Sample hyperparameters from the aggressive regularisation search space for XGBoost."""
    return XGBClassifier(
        n_estimators      = trial.suggest_int("n_estimators",       50,  300),
        max_depth         = trial.suggest_int("max_depth",           2,   5),
        learning_rate     = trial.suggest_float("learning_rate",     0.01, 0.1,  log=True),
        min_child_weight  = trial.suggest_int("min_child_weight",    10,  50),
        subsample         = trial.suggest_float("subsample",         0.5, 0.8),
        colsample_bytree  = trial.suggest_float("colsample_bytree",  0.5, 0.8),
        reg_alpha         = trial.suggest_float("reg_alpha",         0.1, 10.0, log=True),
        reg_lambda        = trial.suggest_float("reg_lambda",        1.0, 20.0, log=True),
        gamma             = trial.suggest_float("gamma",             0.0, 5.0),
        use_label_encoder = False,
        eval_metric       = "logloss",
        random_state      = RANDOM_STATE,
        verbosity         = 0,
        n_jobs            = -1,
    )


def build_lightgbm_model(trial: optuna.Trial) -> LGBMClassifier:
    """Sample hyperparameters from the aggressive regularisation search space for LightGBM."""
    max_depth  = trial.suggest_int("max_depth",  2,  5)
    # num_leaves must be <= 2^max_depth to avoid a misleading config
    max_leaves = min(31, 2 ** max_depth)
    min_leaves = 2
    return LGBMClassifier(
        n_estimators      = trial.suggest_int("n_estimators",        50,  300),
        max_depth         = max_depth,
            num_leaves        = trial.suggest_int("num_leaves",  min_leaves,  max_leaves),
        min_child_samples = trial.suggest_int("min_child_samples",   30,  150),
        learning_rate     = trial.suggest_float("learning_rate",     0.01, 0.1,  log=True),
        reg_alpha         = trial.suggest_float("reg_alpha",         0.1, 10.0, log=True),
        reg_lambda        = trial.suggest_float("reg_lambda",        1.0, 20.0, log=True),
        subsample         = trial.suggest_float("subsample",         0.5, 0.8),
        colsample_bytree  = trial.suggest_float("colsample_bytree",  0.5, 0.8),
        class_weight      = "balanced",
        random_state      = RANDOM_STATE,
        verbose           = -1,
        n_jobs            = -1,
    )


def build_random_forest_model(trial: optuna.Trial) -> RandomForestClassifier:
    """Search space for Random Forest — included because SWDY_CA's winner was RandomForest."""
    return RandomForestClassifier(
        n_estimators   = trial.suggest_int("n_estimators",    100, 500),
        max_depth      = trial.suggest_int("max_depth",         3,  10),
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 20, 100),
        max_features   = trial.suggest_categorical("max_features", ["sqrt", "log2", 0.5]),
        class_weight   = "balanced",
        random_state   = RANDOM_STATE,
        n_jobs         = -1,
    )


MODEL_BUILDERS = {
    "XGBoost":      build_xgboost_model,
    "LightGBM":     build_lightgbm_model,
    "RandomForest": build_random_forest_model,
}

print("✓ Objective builders defined for: XGBoost, LightGBM, RandomForest")

✓ Objective builders defined for: XGBoost, LightGBM, RandomForest


## 7. Optuna Objective and MLflow Callback

In [7]:
mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB.resolve().as_posix()}")

experiment = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT_NAME)
if experiment is None:
    experiment_id = mlflow.create_experiment(MLFLOW_EXPERIMENT_NAME)
    print(f"✓ Created MLflow experiment: '{MLFLOW_EXPERIMENT_NAME}' (id={experiment_id})")
else:
    experiment_id = experiment.experiment_id
    print(f"✓ Resuming MLflow experiment: '{MLFLOW_EXPERIMENT_NAME}' (id={experiment_id})")

print(f"  Tracking URI : {mlflow.get_tracking_uri()}")
print(f"  Experiment ID: {experiment_id}")
print()
print(f"  → Run 'mlflow ui --backend-store-uri sqlite:///{MLFLOW_DB.resolve().as_posix()} --port 5001' to open the dashboard.")

def make_objective(
    model_name: str,
    ticker: str,
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_val: pd.DataFrame,
    y_val: pd.Series,
    parent_run_id: str,
):
    """
    Returns an Optuna objective closure.

    Each trial is logged as a *nested* MLflow run under the parent run for
    this ticker, keeping all trials grouped by ticker in the MLflow UI.

    The objective metric is **validation ROC-AUC**.
    Train ROC-AUC and overfitting gap are logged as diagnostics.
    """
    builder = MODEL_BUILDERS[model_name]

    def objective(trial: optuna.Trial) -> float:
        model = builder(trial)
        model.fit(X_train, y_train)

        train_proba = model.predict_proba(X_train)[:, 1]
        val_proba   = model.predict_proba(X_val)[:, 1]

        train_roc = roc_auc_score(y_train, train_proba)
        val_roc   = roc_auc_score(y_val,   val_proba)
        gap       = round(train_roc - val_roc, 4)

        # Log this trial as a nested MLflow run under the parent ticker run
        with mlflow.start_run(
            experiment_id=experiment_id,
            run_name=f"{ticker}__{model_name}__trial_{trial.number:03d}",
            nested=True,
            tags={"ticker": ticker, "model": model_name, "trial": str(trial.number)},
        ):
            # Log every sampled hyperparameter
            for param_name, param_val in trial.params.items():
                mlflow.log_param(param_name, param_val)
            mlflow.log_param("ticker", ticker)
            mlflow.log_param("model",  model_name)
            mlflow.log_param("trial_number", trial.number)

            # Log metrics
            mlflow.log_metric("train_roc_auc",      round(train_roc, 4))
            mlflow.log_metric("val_roc_auc",        round(val_roc,   4))
            mlflow.log_metric("overfit_gap_roc_auc", gap)

        return val_roc  # Optuna maximises this

    return objective


def progress_callback(study: optuna.Study, trial: optuna.FrozenTrial) -> None:
    """Print a compact progress line every 10 trials."""
    if trial.number % 10 == 0 or trial.number == N_TRIALS - 1:
        best = study.best_value
        current = trial.value if trial.value is not None else float("nan")
        print(
            f"    Trial {trial.number:>3}/{N_TRIALS}  "
            f"current={current:.4f}  "
            f"best_so_far={best:.4f}"
        )


print("✓ Objective factory and progress callback defined.")

✓ Created MLflow experiment: 'EGX_Hyperparameter_Tuning' (id=2)
  Tracking URI : sqlite:///D:/my_projects/Egx-analyst/mlflow.db
  Experiment ID: 2

  → Run 'mlflow ui --backend-store-uri sqlite:///D:/my_projects/Egx-analyst/mlflow.db --port 5001' to open the dashboard.
✓ Objective factory and progress callback defined.


## 8. Main Tuning Loop

For each ticker:
1. Load the winning architecture from `winner_config.json`
2. Open a parent MLflow run to group all trials
3. Run Optuna with the appropriate constrained search space
4. Extract best hyperparameters
5. Retrain final model on full training set with best params
6. Compute full validation metrics + macro-regime H1/H2 diagnostic
7. Compare against baseline from notebook 03

In [ ]:
def compute_full_metrics(
    model: Any,
    X: pd.DataFrame,
    y: pd.Series,
    label: str = "",
) -> Dict[str, float]:
    """Compute the full metric suite for a fitted model on a dataset."""
    if len(y) == 0:
        return {"accuracy": np.nan, "precision": np.nan, "recall": np.nan,
                "f1": np.nan, "roc_auc": np.nan}
    pred  = model.predict(X)
    proba = model.predict_proba(X)[:, 1]
    return {
        "accuracy":  round(accuracy_score(y, pred), 4),
        "precision": round(precision_score(y, pred, zero_division=0), 4),
        "recall":    round(recall_score(y, pred, zero_division=0), 4),
        "f1":        round(f1_score(y, pred, zero_division=0), 4),
        "roc_auc":   round(roc_auc_score(y, proba), 4),
    }


def rebuild_final_model(
    model_name: str,
    best_params: Dict[str, Any],
    scale_pos_weight: Optional[float] = None,
) -> Any:
    """
    Instantiate the winning architecture with optimised hyperparameters.
    Called after Optuna completes to train the final model on the full training set.
    """
    shared_kwargs = dict(random_state=RANDOM_STATE, n_jobs=-1)

    if model_name == "XGBoost":
        xgb_params = dict(best_params)
        if scale_pos_weight is not None:
            xgb_params["scale_pos_weight"] = scale_pos_weight
        return XGBClassifier(
            **xgb_params,
            use_label_encoder=False,
            eval_metric="logloss",
            verbosity=0,
            **shared_kwargs,
        )
    elif model_name == "LightGBM":
        return LGBMClassifier(
            **best_params,
            class_weight="balanced",
            verbose=-1,
            **shared_kwargs,
        )
    elif model_name == "RandomForest":
        return RandomForestClassifier(
            **best_params,
            class_weight="balanced",
            **shared_kwargs,
        )
    else:
        raise ValueError(f"Unknown model_name: {model_name}")


print("✓ Helper functions defined.")

✓ Helper functions defined.


In [9]:
# ── Storage for cross-ticker results dashboard ────────────────────────────────
tuning_results: List[Dict[str, Any]] = []
optimized_hp: Dict[str, Any] = {}

# ── Main loop ─────────────────────────────────────────────────────────────────
for ticker in TICKERS:
    model_name = WINNER_CONFIG[ticker]["model"]
    baseline_val_roc = WINNER_CONFIG[ticker]["val_roc_auc"]
    baseline_val_f1  = WINNER_CONFIG[ticker]["val_f1"]

    print(f"\n{'━'*65}")
    print(f"  TICKER : {ticker}")
    print(f"  MODEL  : {model_name}")
    print(f"  BASELINE (notebook 03)  val_roc_auc={baseline_val_roc:.4f}  val_f1={baseline_val_f1:.4f}")
    print(f"{'━'*65}")

    # ── Load data ────────────────────────────────────────────────────────────
    (
        X_train, y_train,
        X_val,   y_val,
        X_val_h1, y_val_h1,
        X_val_h2, y_val_h2,
        val_index,
    ) = load_ticker_splits(ticker)
    print(f"  Train={len(X_train)}  Val={len(X_val)}  "
          f"Val-H1={len(X_val_h1)} (pre-devaluation)  "
          f"Val-H2={len(X_val_h2)} (post-devaluation)")

    # ── Open parent MLflow run to group all trials ────────────────────────────
    with mlflow.start_run(
        experiment_id=experiment_id,
        run_name=f"{ticker}__{model_name}__optuna",
        tags={"ticker": ticker, "model": model_name, "stage": "tuning"},
    ) as parent_run:

        mlflow.log_param("ticker",              ticker)
        mlflow.log_param("model",               model_name)
        mlflow.log_param("n_trials",            N_TRIALS)
        mlflow.log_param("train_rows",          len(X_train))
        mlflow.log_param("val_rows",            len(X_val))
        mlflow.log_param("baseline_val_roc_auc", baseline_val_roc)

        # ── Run Optuna study ─────────────────────────────────────────────────
        print(f"\n  Running Optuna ({N_TRIALS} trials) ...")
        study = optuna.create_study(
            direction="maximize",
            sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
            pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=5),
        )
        objective_fn = make_objective(
            model_name=model_name,
            ticker=ticker,
            X_train=X_train,
            y_train=y_train,
            X_val=X_val,
            y_val=y_val,
            parent_run_id=parent_run.info.run_id,
        )
        study.optimize(
            objective_fn,
            n_trials=N_TRIALS,
            callbacks=[progress_callback],
            show_progress_bar=False,
        )

        best_params = study.best_params
        best_val_roc_optuna = study.best_value
        print(f"\n  ✓ Optuna complete. Best trial val_roc_auc = {best_val_roc_optuna:.4f}")
        print(f"  Best hyperparameters:")
        for k, v in best_params.items():
            print(f"    {k:<22} = {v}")

        # ── Retrain final model with best params on full train set ───────────
        print(f"\n  Retraining final model with best params ...")
        final_model = rebuild_final_model(model_name, best_params)
        final_model.fit(X_train, y_train)

        # ── Full validation metrics ───────────────────────────────────────────
        train_metrics  = compute_full_metrics(final_model, X_train,  y_train)
        val_metrics    = compute_full_metrics(final_model, X_val,    y_val)
        val_h1_metrics = compute_full_metrics(final_model, X_val_h1, y_val_h1)
        val_h2_metrics = compute_full_metrics(final_model, X_val_h2, y_val_h2)

        overfit_gap = round(train_metrics["roc_auc"] - val_metrics["roc_auc"], 4)
        roc_improvement = round(val_metrics["roc_auc"] - baseline_val_roc, 4)

        # ── Log final metrics to parent run ──────────────────────────────────
        mlflow.log_metric("final_train_roc_auc",    train_metrics["roc_auc"])
        mlflow.log_metric("final_val_roc_auc",      val_metrics["roc_auc"])
        mlflow.log_metric("final_val_f1",           val_metrics["f1"])
        mlflow.log_metric("final_val_precision",    val_metrics["precision"])
        mlflow.log_metric("final_val_recall",       val_metrics["recall"])
        mlflow.log_metric("final_overfit_gap",      overfit_gap)
        mlflow.log_metric("val_h1_roc_auc",         val_h1_metrics["roc_auc"])
        mlflow.log_metric("val_h2_roc_auc",         val_h2_metrics["roc_auc"])
        mlflow.log_metric("roc_auc_improvement",    roc_improvement)
        mlflow.log_metric("baseline_val_roc_auc",   baseline_val_roc)

        for k, v in best_params.items():
            mlflow.log_param(f"best_{k}", v)

        # Log the final model
        if model_name == "XGBoost":
            mlflow.xgboost.log_model(final_model, name="optimised_model")
        elif model_name == "LightGBM":
            mlflow.lightgbm.log_model(final_model, name="optimised_model")
        else:
            mlflow.sklearn.log_model(final_model, name="optimised_model")

        parent_run_id = parent_run.info.run_id

    # ── Macro-regime diagnostic print ────────────────────────────────────────
    h1_roc = val_h1_metrics["roc_auc"]
    h2_roc = val_h2_metrics["roc_auc"]
    regime_delta = round(h1_roc - h2_roc, 4)

    print(f"\n  {'─'*60}")
    print(f"  MACRO-REGIME DIAGNOSTIC (2024 validation split)")
    print(f"  {'─'*60}")
    print(f"  H1 (Jan–Jun 2024, pre-devaluation)  val_roc_auc = {h1_roc:.4f}")
    print(f"  H2 (Jul–Dec 2024, post-devaluation) val_roc_auc = {h2_roc:.4f}")
    print(f"  H1 − H2 delta = {regime_delta:+.4f}", end="  ")
    if abs(regime_delta) < 0.03:
        print("→ No significant regime effect detected.")
    elif regime_delta > 0.03:
        print("→ ⚠️  Performance drops post-devaluation: likely macro-regime shift, not model failure.")
    else:
        print("→ ✓ Performance improves post-devaluation (model adapts to new regime).")

    # ── Summary print ─────────────────────────────────────────────────────────
    gap_status = "✓" if overfit_gap < 0.10 else ("⚠" if overfit_gap < 0.20 else "✗")
    roc_status = "✓" if val_metrics["roc_auc"] >= 0.56 else "⚠"

    print(f"\n  {'─'*60}")
    print(f"  TUNING RESULTS SUMMARY")
    print(f"  {'─'*60}")
    print(f"  {'Metric':<28} {'Baseline':>10} {'Optimised':>11} {'Delta':>8}")
    print(f"  {'─'*60}")
    print(f"  {'val_roc_auc':<28} {baseline_val_roc:>10.4f} {val_metrics['roc_auc']:>11.4f} {roc_improvement:>+8.4f} {roc_status}")
    print(f"  {'val_f1':<28} {baseline_val_f1:>10.4f} {val_metrics['f1']:>11.4f} {val_metrics['f1']-baseline_val_f1:>+8.4f}")
    print(f"  {'val_precision':<28} {'—':>10} {val_metrics['precision']:>11.4f}")
    print(f"  {'val_recall':<28} {'—':>10} {val_metrics['recall']:>11.4f}")
    print(f"  {'overfit_gap (train−val AUC)':<28} {'—':>10} {overfit_gap:>+11.4f} {gap_status} (target < 0.10)")
    print(f"  {'─'*60}")

    # ── Collect results ───────────────────────────────────────────────────────
    tuning_results.append({
        "ticker":               ticker,
        "model":                model_name,
        "baseline_val_roc_auc": baseline_val_roc,
        "baseline_val_f1":      baseline_val_f1,
        "tuned_train_roc_auc":  train_metrics["roc_auc"],
        "tuned_val_roc_auc":    val_metrics["roc_auc"],
        "tuned_val_f1":         val_metrics["f1"],
        "tuned_val_precision":  val_metrics["precision"],
        "tuned_val_recall":     val_metrics["recall"],
        "overfit_gap":          overfit_gap,
        "roc_improvement":      roc_improvement,
        "val_h1_roc_auc":       h1_roc,
        "val_h2_roc_auc":       h2_roc,
        "h1_h2_regime_delta":   regime_delta,
        "n_optuna_trials":       N_TRIALS,
        "parent_run_id":         parent_run_id,
    })

    optimized_hp[ticker] = {
        "model":      model_name,
        "params":     best_params,
        "tuned_val_roc_auc": round(val_metrics["roc_auc"], 4),
        "tuned_val_f1":      round(val_metrics["f1"], 4),
        "overfit_gap":       overfit_gap,
        "parent_run_id":     parent_run_id,
    }


print(f"\n\n{'═'*65}")
print(f"  ✓ Tuning complete for all {len(TICKERS)} tickers.")
print(f"{'═'*65}")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  TICKER : COMI_CA
  MODEL  : XGBoost
  BASELINE (notebook 03)  val_roc_auc=0.5450  val_f1=0.5517
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Train=1151  Val=237  Val-H1=110 (pre-devaluation)  Val-H2=127 (post-devaluation)

  Running Optuna (75 trials) ...
    Trial   0/75  current=0.4944  best_so_far=0.4944
    Trial  10/75  current=0.4760  best_so_far=0.4988
    Trial  20/75  current=0.4994  best_so_far=0.5028
    Trial  30/75  current=0.5066  best_so_far=0.5066
    Trial  40/75  current=0.5027  best_so_far=0.5066
    Trial  50/75  current=0.4824  best_so_far=0.5103
    Trial  60/75  current=0.5094  best_so_far=0.5267
    Trial  70/75  current=0.5011  best_so_far=0.5267
    Trial  74/75  current=0.5037  best_so_far=0.5267

  ✓ Optuna complete. Best trial val_roc_auc = 0.5267
  Best hyperparameters:
    n_estimators           = 200
    max_depth              = 4
    learning_rate          = 0.08

## 9. Executive Dashboard — Baseline vs Optimised

In [10]:
results_df = pd.DataFrame(tuning_results)

# ── Styled comparison table ───────────────────────────────────────────────────
dashboard_cols = [
    "ticker", "model",
    "baseline_val_roc_auc", "tuned_val_roc_auc", "roc_improvement",
    "tuned_val_f1",
    "overfit_gap",
    "val_h1_roc_auc", "val_h2_roc_auc", "h1_h2_regime_delta",
]

display(
    results_df[dashboard_cols]
    .style
    .background_gradient(subset=["tuned_val_roc_auc", "tuned_val_f1"], cmap="RdYlGn")
    .background_gradient(subset=["overfit_gap"], cmap="RdYlGn_r")      # lower = greener
    .background_gradient(subset=["roc_improvement"], cmap="RdYlGn")    # positive = greener
    .background_gradient(subset=["h1_h2_regime_delta"], cmap="RdYlGn_r")
    .format({
        "baseline_val_roc_auc":  "{:.4f}",
        "tuned_val_roc_auc":     "{:.4f}",
        "roc_improvement":       "{:+.4f}",
        "tuned_val_f1":          "{:.4f}",
        "overfit_gap":           "{:+.4f}",
        "val_h1_roc_auc":        "{:.4f}",
        "val_h2_roc_auc":        "{:.4f}",
        "h1_h2_regime_delta":    "{:+.4f}",
    })
    .set_caption(
        "Tuning Dashboard — green = better | "
        "overfit_gap target < +0.10 | "
        "large h1_h2_regime_delta → macro-regime shift, not model failure"
    )
)

# ── Plain text summary for quick reading ─────────────────────────────────────
print("\nExecutive Summary")
print("=" * 70)
for _, row in results_df.iterrows():
    gap_status = "✓" if row["overfit_gap"] < 0.10 else ("⚠" if row["overfit_gap"] < 0.20 else "✗")
    roc_status = "✓" if row["tuned_val_roc_auc"] >= 0.56 else "⚠"
    print(
        f"  {row['ticker']:<12} {row['model']:<14}  "
        f"ROC: {row['baseline_val_roc_auc']:.4f} → {row['tuned_val_roc_auc']:.4f} "
        f"({row['roc_improvement']:+.4f}) {roc_status}  "
        f"gap={row['overfit_gap']:+.4f} {gap_status}"
    )
print("=" * 70)

,ticker,model,baseline_val_roc_auc,tuned_val_roc_auc,roc_improvement,tuned_val_f1,overfit_gap,val_h1_roc_auc,val_h2_roc_auc,h1_h2_regime_delta
0,COMI_CA,XGBoost,0.5450,0.5267,-0.0183,0.5249,+0.2352,0.4987,0.5447,-0.0460
1,HRHO_CA,XGBoost,0.5496,0.5646,+0.0150,0.0000,-0.0244,0.5450,0.5754,-0.0304
2,ORWE_CA,LightGBM,0.5489,0.5895,+0.0406,0.5630,+0.1789,0.5725,0.5910,-0.0185
3,SWDY_CA,RandomForest,0.5133,0.5342,+0.0209,0.6132,+0.1381,0.4530,0.5674,-0.1144
4,TMGH_CA,LightGBM,0.5209,0.5264,+0.0055,0.4519,+0.3461,0.5349,0.5299,+0.0050



Executive Summary
  COMI_CA      XGBoost         ROC: 0.5450 → 0.5267 (-0.0183) ⚠  gap=+0.2352 ✗
  HRHO_CA      XGBoost         ROC: 0.5496 → 0.5646 (+0.0150) ✓  gap=-0.0244 ✓
  ORWE_CA      LightGBM        ROC: 0.5489 → 0.5895 (+0.0406) ✓  gap=+0.1789 ⚠
  SWDY_CA      RandomForest    ROC: 0.5133 → 0.5342 (+0.0209) ⚠  gap=+0.1381 ⚠
  TMGH_CA      LightGBM        ROC: 0.5209 → 0.5264 (+0.0055) ⚠  gap=+0.3461 ✗


## 10. Visualisations

In [11]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
x = np.arange(len(results_df))
ticker_labels = results_df["ticker"].tolist()
width = 0.35

# ── Panel 1: Baseline vs Tuned ROC-AUC ───────────────────────────────────────
ax = axes[0]
ax.bar(x - width/2, results_df["baseline_val_roc_auc"], width, label="Baseline",
       color="#d9534f", edgecolor="black", alpha=0.85)
ax.bar(x + width/2, results_df["tuned_val_roc_auc"],   width, label="Tuned (Optuna)",
       color="#5cb85c", edgecolor="black", alpha=0.85)
ax.axhline(0.56, linestyle="--", color="navy",   linewidth=1.2, label="Target lower (0.56)")
ax.axhline(0.50, linestyle=":",  color="grey",   linewidth=1.0, label="Random (0.50)")
ax.set_xticks(x); ax.set_xticklabels(ticker_labels, rotation=20, ha="right")
ax.set_ylabel("Validation ROC-AUC")
ax.set_title("Baseline vs Tuned\nValidation ROC-AUC", fontweight="bold")
ax.legend(fontsize=7); ax.set_ylim(0.44, 0.72)

# ── Panel 2: Overfitting gap ──────────────────────────────────────────────────
ax = axes[1]
colors = ["#5cb85c" if g < 0.10 else ("#f0ad4e" if g < 0.20 else "#d9534f")
          for g in results_df["overfit_gap"]]
bars = ax.bar(x, results_df["overfit_gap"], color=colors, edgecolor="black", alpha=0.9)
ax.axhline(0.10, linestyle="--", color="green", linewidth=1.2, label="Target ceiling (0.10)")
ax.axhline(0.20, linestyle=":",  color="orange", linewidth=1.0, label="Warning (0.20)")
ax.set_xticks(x); ax.set_xticklabels(ticker_labels, rotation=20, ha="right")
ax.set_ylabel("Overfit Gap (train AUC − val AUC)")
ax.set_title("Overfitting Gap After Tuning\n(green < 0.10 = target met)", fontweight="bold")
ax.legend(fontsize=7)
for bar, val in zip(bars, results_df["overfit_gap"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f"{val:+.3f}", ha="center", va="bottom", fontsize=8)

# ── Panel 3: Macro-regime H1 vs H2 ───────────────────────────────────────────
ax = axes[2]
ax.bar(x - width/2, results_df["val_h1_roc_auc"], width, label="H1 (pre-devaluation)",
       color="#5bc0de", edgecolor="black", alpha=0.85)
ax.bar(x + width/2, results_df["val_h2_roc_auc"], width, label="H2 (post-devaluation)",
       color="#f0ad4e", edgecolor="black", alpha=0.85)
ax.axhline(0.50, linestyle=":", color="grey", linewidth=1.0, label="Random (0.50)")
ax.set_xticks(x); ax.set_xticklabels(ticker_labels, rotation=20, ha="right")
ax.set_ylabel("Validation ROC-AUC")
ax.set_title("Macro-Regime Diagnostic\nH1 (Jan–Jun) vs H2 (Jul–Dec) 2024", fontweight="bold")
ax.legend(fontsize=7); ax.set_ylim(0.40, 0.75)

plt.suptitle(
    f"EGX Directional Prediction — Hyperparameter Tuning Results ({N_TRIALS} Optuna Trials)",
    fontsize=12, fontweight="bold", y=1.02
)
plt.tight_layout()

chart_path = ARTIFACTS_DIR / "tuning_dashboard.png"
fig.savefig(chart_path, dpi=130, bbox_inches="tight")
plt.show()
print(f"Dashboard chart saved → {chart_path}")

Dashboard chart saved → tuning_artifacts\tuning_dashboard.png


## 11. Export Optimised Hyperparameters

Saves `models/optimized_hyperparameters.json` — the **tuning contract** consumed by notebook 05 (final test-set evaluation) and by `src/train.py` (production training). This file contains the exact hyperparameters that produced the validated results above.

In [12]:
# Ensure all values are JSON-serialisable (numpy types → native Python)
def make_serialisable(obj: Any) -> Any:
    if isinstance(obj, dict):
        return {k: make_serialisable(v) for k, v in obj.items()}
    elif isinstance(obj, (np.integer,)):
        return int(obj)
    elif isinstance(obj, (np.floating,)):
        return float(obj)
    elif isinstance(obj, (np.ndarray,)):
        return obj.tolist()
    return obj


serialisable_hp = make_serialisable(optimized_hp)
OPTIMIZED_HP_PATH.write_text(json.dumps(serialisable_hp, indent=2))

print(f"✓ Optimised hyperparameters saved → {OPTIMIZED_HP_PATH.resolve()}")
print()
print(json.dumps(serialisable_hp, indent=2))

✓ Optimised hyperparameters saved → D:\my_projects\Egx-analyst\notebooks\models\optimized_hyperparameters.json

{
  "COMI_CA": {
    "model": "XGBoost",
    "params": {
      "n_estimators": 200,
      "max_depth": 4,
      "learning_rate": 0.08948638193179499,
      "min_child_weight": 18,
      "subsample": 0.7977274633922923,
      "colsample_bytree": 0.7398766541693323,
      "reg_alpha": 3.4805749095314273,
      "reg_lambda": 6.446206068296873,
      "gamma": 1.6339637005138417
    },
    "tuned_val_roc_auc": 0.5267,
    "tuned_val_f1": 0.5249,
    "overfit_gap": 0.2352,
    "parent_run_id": "c425ffee12de47cfb587f17b50c0ac21"
  },
  "HRHO_CA": {
    "model": "XGBoost",
    "params": {
      "n_estimators": 50,
      "max_depth": 2,
      "learning_rate": 0.01589877345097281,
      "min_child_weight": 41,
      "subsample": 0.528662524206753,
      "colsample_bytree": 0.6834523009914724,
      "reg_alpha": 6.842575519287799,
      "reg_lambda": 3.8924761914366317,
      "gamma": 3

## 12. MLflow Results Verification

Query the tracking store to confirm every parent run was logged and retrieve final metrics.

In [13]:
mlflow_runs = mlflow.search_runs(
    experiment_ids=[experiment_id],
    filter_string="tags.stage = 'tuning'",
    order_by=["metrics.final_val_roc_auc DESC"],
)

print(f"MLflow parent runs logged: {len(mlflow_runs)} (expected {len(TICKERS)})")
if len(mlflow_runs) > 0:
    view_cols = [
        c for c in [
            "tags.ticker", "tags.model",
            "metrics.baseline_val_roc_auc",
            "metrics.final_val_roc_auc",
            "metrics.final_val_f1",
            "metrics.final_overfit_gap",
            "metrics.val_h1_roc_auc",
            "metrics.val_h2_roc_auc",
        ]
        if c in mlflow_runs.columns
    ]
    display(mlflow_runs[view_cols].rename(columns=lambda c: c.replace("metrics.", "").replace("tags.", "")))

MLflow parent runs logged: 5 (expected 5)


,ticker,model,baseline_val_roc_auc,final_val_roc_auc,final_val_f1,final_overfit_gap,val_h1_roc_auc,val_h2_roc_auc
0,ORWE_CA,LightGBM,0.5489,0.5895,0.5630,0.1789,0.5725,0.5910
1,HRHO_CA,XGBoost,0.5496,0.5646,0.0000,-0.0244,0.5450,0.5754
2,SWDY_CA,RandomForest,0.5133,0.5342,0.6132,0.1381,0.4530,0.5674
3,COMI_CA,XGBoost,0.5450,0.5267,0.5249,0.2352,0.4987,0.5447
4,TMGH_CA,LightGBM,0.5209,0.5264,0.4519,0.3461,0.5349,0.5299
